In [ ]:
import pandas as pd
df = pd.read_parquet("../data/processed/karachi_2000_2020_raw.parquet")
print(df.shape)

In [ ]:
# Which columns actually have data? (we choose columns from this in 1.3)
filled = df.notna().mean().sort_values(ascending=False)
print(filled[filled >= 0.05].round(2).to_string())
print()
print("Columns under 5% filled:", filled[filled < 0.05].index.tolist())

In [ ]:
# The main categories
for col in ["gname", "attacktype1_txt", "targtype1_txt", "weaptype1_txt"]:
    print(df[col].value_counts().head(10), "\n")

In [ ]:
# How good is the text?
lengths = df["summary"].dropna().str.len()
print(lengths.describe().round(0))
print("Duplicate summaries:", df["summary"].duplicated().sum())

In [ ]:
# Actors, harm, and the existing event links
print("Unknown group share:", round(df["gname"].eq("Unknown").mean(), 2))
print(df[["nkill", "nwound"]].describe().round(1))
print("Events with a 'related' link:", df["related"].notna().sum())

In [ ]:
# 1. Coordinate precision: if many events share one point, "same area" can't come from coordinates
print(df["specificity"].value_counts())
print("Distinct coordinates:", df[["latitude", "longitude"]].drop_duplicates().shape[0])
print(df.groupby(["latitude", "longitude"]).size().sort_values(ascending=False).head(5))

In [ ]:
# 2. What does the location text look like?
print(df["location"].dropna().sample(10, random_state=1).to_string())

In [ ]:
# 3. What does the GTD "related" column contain?
print(df["related"].dropna().head(5).tolist())

In [ ]:
# 4. Generic group names hiding among the groups
print(df["gname"].value_counts().head(40).to_string())

In [ ]:
# 5. Events GTD itself doubts are terrorism
print(df["doubtterr"].value_counts())

In [ ]:
# 6. The 7 duplicate summaries
dups = df[df["summary"].duplicated(keep=False)].sort_values("summary")
print(dups.assign(summary=dups["summary"].str[:80])[["eventid", "iyear", "gname", "summary"]].to_string())

In [ ]:
for col in ["claimed", "guncertain1", "suicide", "success", "multiple"]:
    print(col, df[col].value_counts(dropna=False).to_dict())

print("imonth == 0:", (df["imonth"] == 0).sum(), "| iday == 0:", (df["iday"] == 0).sum())
print("negative nkill / nwound:", (df["nkill"] < 0).sum(), (df["nwound"] < 0).sum())
print("Summaries with a date prefix:", df["summary"].str.match(r"^\d{2}/\d{2}/\d{4}:").mean().round(2))